# NumpyNet — Neural Network from Scratch on Fashion MNIST

A complete neural network library implemented using only NumPy — no PyTorch,
TensorFlow, or autograd. Every forward pass, backward pass, and parameter
update is derived and coded manually.

**Architecture:** 784 → 128 → 64 → 10  
**Implemented from scratch:** Dense layers, ReLU, Softmax, Categorical
Cross-Entropy (with combined Softmax backward for speed), Dropout, L1/L2
regularization, Adam optimizer, mini-batch training with accumulated
metrics.

**Dataset:** Fashion MNIST (60,000 train / 10,000 test, 28×28 grayscale,
10 classes)

**Result:** 87.9% validation accuracy after 10 epochs.

In [1]:
import numpy as np
import os
import urllib.request
from zipfile import ZipFile
import cv2

np.random.seed(0)



## Layers

`Layer_Dense` implements the forward pass (`y = Wx + b`) and backward pass
(gradients on weights, biases, and inputs via the chain rule), including
L1/L2 regularization gradients. `Layer_Dropout` implements inverted dropout
— scales surviving activations during training, passes through unchanged
during inference.

In [2]:
# LAYERS
# ══════════════════════════════════════════════════════════════════════

class Layer_Dense:


    def __init__(self, n_inputs, n_neurons,
                 weight_regularizer_l1=0, weight_regularizer_l2=0,
                 bias_regularizer_l1=0, bias_regularizer_l2=0):

        self.weights = 0.1 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1 = bias_regularizer_l1
        self.bias_regularizer_l2 = bias_regularizer_l2

    def forward(self, inputs, training):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):

        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)


        if self.weight_regularizer_l1 > 0:
            dL1 = np.ones_like(self.weights)
            dL1[self.weights < 0] = -1
            self.dweights += self.weight_regularizer_l1 * dL1

        if self.bias_regularizer_l1 > 0:
            dL1 = np.ones_like(self.biases)
            dL1[self.biases < 0] = -1
            self.dbiases += self.bias_regularizer_l1 * dL1

        if self.weight_regularizer_l2 > 0:
            self.dweights += 2 * self.weight_regularizer_l2 * self.weights

        if self.bias_regularizer_l2 > 0:
            self.dbiases += 2 * self.bias_regularizer_l2 * self.biases

        self.dinputs = np.dot(dvalues, self.weights.T)


class Layer_Dropout:


    def __init__(self, rate):

        self.rate = 1 - rate

    def forward(self, inputs, training):
        self.inputs = inputs

        if not training:
            self.output = inputs.copy()
            return

        self.binary_mask = np.random.binomial(1, self.rate, size=inputs.shape) / self.rate
        self.output = inputs * self.binary_mask

    def backward(self, dvalues):
        self.dinputs = dvalues * self.binary_mask


class Layer_Input:


    def forward(self, inputs, training):
        self.output = inputs



## Activations

`Activation_ReLU` zeroes gradients where the pre-activation was negative.
`Activation_Softmax` computes the full Jacobian for its backward pass —
used here for verification, but bypassed during training in favor of the
faster combined Softmax+CrossEntropy gradient below.

In [3]:
 #ACTIVATIONS
# ══════════════════════════════════════════════════════════════════════

class Activation_ReLU:
    def forward(self, inputs, training):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

    def predictions(self, outputs):
        return outputs


class Activation_Softmax:
    def forward(self, inputs, training):
        self.inputs = inputs
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = exp_values / np.sum(exp_values, axis=1, keepdims=True)

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)
        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1, 1)
            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)

    def predictions(self, outputs):
        return np.argmax(outputs, axis=1)



## Loss

`Loss_CategoricalCrossentropy` computes the standard cross-entropy loss
and gradient. `Activation_Softmax_Loss_CategoricalCrossentropy` implements
the combined backward pass — the gradient simplifies to `predictions -
true_labels`, which is ~7x faster than computing the Softmax Jacobian and
cross-entropy gradient separately.

In [4]:

# ══════════════════════════════════════════════════════════════════════
# LOSS
# ══════════════════════════════════════════════════════════════════════

class Loss:
    def remember_trainable_layers(self, trainable_layers):
        self.trainable_layers = trainable_layers

    def regularization_loss(self):
        regularization_loss = 0
        for layer in self.trainable_layers:
            if layer.weight_regularizer_l1 > 0:
                regularization_loss += layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))
            if layer.weight_regularizer_l2 > 0:
                regularization_loss += layer.weight_regularizer_l2 * np.sum(layer.weights * layer.weights)
            if layer.bias_regularizer_l1 > 0:
                regularization_loss += layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))
            if layer.bias_regularizer_l2 > 0:
                regularization_loss += layer.bias_regularizer_l2 * np.sum(layer.biases * layer.biases)
        return regularization_loss

    def calculate(self, output, y, *, include_regularization=False):
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)

        self.accumulated_sum += np.sum(sample_losses)
        self.accumulated_count += len(sample_losses)

        if not include_regularization:
            return data_loss
        return data_loss, self.regularization_loss()

    def calculate_accumulated(self, *, include_regularization=False):
        data_loss = self.accumulated_sum / self.accumulated_count
        if not include_regularization:
            return data_loss
        return data_loss, self.regularization_loss()

    def new_pass(self):
        self.accumulated_sum = 0
        self.accumulated_count = 0


class Activation_Softmax_Loss_CategoricalCrossentropy:
    """Combined Softmax + CrossEntropy backward pass (7x faster gradient)."""

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs / samples


class Loss_CategoricalCrossentropy(Loss):
    def forward(self, y_pred, y_true):
        samples = len(y_pred)
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)

        return -np.log(correct_confidences)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        labels = len(dvalues[0])

        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]

        self.dinputs = -y_true / dvalues
        self.dinputs = self.dinputs / samples



## Optimizer — Adam

Combines momentum (first moment) and adaptive per-parameter learning rates
(second moment, RMSProp-style), with bias correction for both — critical
in early training steps when the moment estimates are still near zero.

In [5]:

# ══════════════════════════════════════════════════════════════════════
# OPTIMIZER - Adam
# ══════════════════════════════════════════════════════════════════════

class Optimizer_Adam:
    def __init__(self, learning_rate=0.001, decay=0., epsilon=1e-7,
                 beta_1=0.9, beta_2=0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * \
                (1. / (1. + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        layer.weight_momentums = self.beta_1 * layer.weight_momentums + \
            (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + \
            (1 - self.beta_1) * layer.dbiases


        weight_momentums_corrected = layer.weight_momentums / \
            (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected = layer.bias_momentums / \
            (1 - self.beta_1 ** (self.iterations + 1))

        layer.weight_cache = self.beta_2 * layer.weight_cache + \
            (1 - self.beta_2) * layer.dweights ** 2
        layer.bias_cache = self.beta_2 * layer.bias_cache + \
            (1 - self.beta_2) * layer.dbiases ** 2

        weight_cache_corrected = layer.weight_cache / \
            (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected = layer.bias_cache / \
            (1 - self.beta_2 ** (self.iterations + 1))


        layer.weights += -self.current_learning_rate * \
            weight_momentums_corrected / (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases += -self.current_learning_rate * \
            bias_momentums_corrected / (np.sqrt(bias_cache_corrected) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1



In [6]:

# ══════════════════════════════════════════════════════════════════════
# ACCURACY
# ══════════════════════════════════════════════════════════════════════

class Accuracy_Categorical:
    def init(self, y):
        pass

    def compare(self, predictions, y):
        if len(y.shape) == 2:
            y = np.argmax(y, axis=1)
        return predictions == y

    def calculate(self, predictions, y):
        comparisons = self.compare(predictions, y)
        accuracy = np.mean(comparisons)
        self.accumulated_sum += np.sum(comparisons)
        self.accumulated_count += len(comparisons)
        return accuracy

    def calculate_accumulated(self):
        return self.accumulated_sum / self.accumulated_count

    def new_pass(self):
        self.accumulated_sum = 0
        self.accumulated_count = 0



## Model

Wires layers together via `.prev`/`.next` references so forward and
backward passes are uniform loops regardless of architecture. Handles
batching, accumulated loss/accuracy across an epoch, and validation.

In [7]:

# ══════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════

class Model:
    def __init__(self):
        self.layers = []
        self.softmax_classifier_output = None

    def add(self, layer):
        self.layers.append(layer)

    def set(self, *, loss, optimizer, accuracy):
        self.loss = loss
        self.optimizer = optimizer
        self.accuracy = accuracy

    def finalize(self):
        self.input_layer = Layer_Input()
        layer_count = len(self.layers)
        self.trainable_layers = []

        for i in range(layer_count):
            if i == 0:
                self.layers[i].prev = self.input_layer
                self.layers[i].next = self.layers[i + 1]
            elif i < layer_count - 1:
                self.layers[i].prev = self.layers[i - 1]
                self.layers[i].next = self.layers[i + 1]
            else:
                self.layers[i].prev = self.layers[i - 1]
                self.layers[i].next = self.loss
                self.output_layer_activation = self.layers[i]

            if hasattr(self.layers[i], 'weights'):
                self.trainable_layers.append(self.layers[i])

        self.loss.remember_trainable_layers(self.trainable_layers)

        if isinstance(self.layers[-1], Activation_Softmax) and \
           isinstance(self.loss, Loss_CategoricalCrossentropy):
            self.softmax_classifier_output = Activation_Softmax_Loss_CategoricalCrossentropy()

    def forward(self, X, training):
        self.input_layer.forward(X, training)
        for layer in self.layers:
            layer.forward(layer.prev.output, training)
        return layer.output

    def backward(self, output, y):
        if self.softmax_classifier_output is not None:
            self.softmax_classifier_output.backward(output, y)
            self.layers[-1].dinputs = self.softmax_classifier_output.dinputs
            for layer in reversed(self.layers[:-1]):
                layer.backward(layer.next.dinputs)
            return

        self.loss.backward(output, y)
        for layer in reversed(self.layers):
            layer.backward(layer.next.dinputs)

    def train(self, X, y, *, epochs=1, batch_size=None, print_every=1, validation_data=None):
        self.accuracy.init(y)
        train_steps = 1

        if validation_data is not None:
            validation_steps = 1
            X_val, y_val = validation_data

        if batch_size is not None:
            train_steps = len(X) // batch_size
            if train_steps * batch_size < len(X):
                train_steps += 1
            if validation_data is not None:
                validation_steps = len(X_val) // batch_size
                if validation_steps * batch_size < len(X_val):
                    validation_steps += 1

        for epoch in range(1, epochs + 1):
            print(f'epoch: {epoch}')
            self.loss.new_pass()
            self.accuracy.new_pass()

            for step in range(train_steps):
                if batch_size is None:
                    batch_X, batch_y = X, y
                else:
                    batch_X = X[step * batch_size:(step + 1) * batch_size]
                    batch_y = y[step * batch_size:(step + 1) * batch_size]

                output = self.forward(batch_X, training=True)

                data_loss, regularization_loss = self.loss.calculate(
                    output, batch_y, include_regularization=True)
                loss = data_loss + regularization_loss

                predictions = self.output_layer_activation.predictions(output)
                accuracy = self.accuracy.calculate(predictions, batch_y)

                self.backward(output, batch_y)

                self.optimizer.pre_update_params()
                for layer in self.trainable_layers:
                    self.optimizer.update_params(layer)
                self.optimizer.post_update_params()

                if not step % print_every or step == train_steps - 1:
                    print(f'  step: {step}, acc: {accuracy:.3f}, ' +
                          f'loss: {loss:.3f} (data_loss: {data_loss:.3f}, ' +
                          f'reg_loss: {regularization_loss:.3f}), ' +
                          f'lr: {self.optimizer.current_learning_rate}')

            epoch_data_loss, epoch_regularization_loss = self.loss.calculate_accumulated(
                include_regularization=True)
            epoch_loss = epoch_data_loss + epoch_regularization_loss
            epoch_accuracy = self.accuracy.calculate_accumulated()
            print(f'  training,   acc: {epoch_accuracy:.3f}, ' +
                  f'loss: {epoch_loss:.3f} (data_loss: {epoch_data_loss:.3f}, ' +
                  f'reg_loss: {epoch_regularization_loss:.3f}), ' +
                  f'lr: {self.optimizer.current_learning_rate}')

            if validation_data is not None:
                self.loss.new_pass()
                self.accuracy.new_pass()

                for step in range(validation_steps):
                    if batch_size is None:
                        batch_X, batch_y = X_val, y_val
                    else:
                        batch_X = X_val[step * batch_size:(step + 1) * batch_size]
                        batch_y = y_val[step * batch_size:(step + 1) * batch_size]

                    output = self.forward(batch_X, training=False)
                    self.loss.calculate(output, batch_y)
                    predictions = self.output_layer_activation.predictions(output)
                    self.accuracy.calculate(predictions, batch_y)

                validation_loss = self.loss.calculate_accumulated()
                validation_accuracy = self.accuracy.calculate_accumulated()
                print(f'  validation, acc: {validation_accuracy:.3f}, loss: {validation_loss:.3f}')

    def predict(self, X, *, batch_size=None):
        prediction_steps = 1
        if batch_size is not None:
            prediction_steps = len(X) // batch_size
            if prediction_steps * batch_size < len(X):
                prediction_steps += 1

        output = []
        for step in range(prediction_steps):
            if batch_size is None:
                batch_X = X
            else:
                batch_X = X[step * batch_size:(step + 1) * batch_size]

            batch_output = self.forward(batch_X, training=False)
            output.append(batch_output)

        return np.vstack(output)



## Data Loading

Downloads Fashion MNIST, loads images via OpenCV, scales pixel values to
[-1, 1], flattens 28×28 images to 784-dimensional vectors, and shuffles
the training set (critical — without shuffling, the model sees one class
at a time per batch and fails to generalize, as covered in NNFS chapter 19).

In [8]:

# ══════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════

def download_fashion_mnist():
    URL = 'https://nnfs.io/datasets/fashion_mnist_images.zip'
    FILE = 'fashion_mnist_images.zip'
    FOLDER = 'fashion_mnist_images'

    if not os.path.isfile(FILE):
        print(f'Downloading {URL}...')
        urllib.request.urlretrieve(URL, FILE)

    if not os.path.isdir(FOLDER):
        print('Unzipping images...')
        with ZipFile(FILE) as zip_images:
            zip_images.extractall(FOLDER)

    print('Data ready.')


def load_mnist_dataset(dataset, path):
    labels = os.listdir(os.path.join(path, dataset))
    X = []
    y = []

    for label in labels:
        for file in os.listdir(os.path.join(path, dataset, label)):
            image = cv2.imread(os.path.join(path, dataset, label, file), cv2.IMREAD_UNCHANGED)
            X.append(image)
            y.append(label)

    return np.array(X), np.array(y).astype('uint8')


def create_data_mnist(path):
    X, y = load_mnist_dataset('train', path)
    X_test, y_test = load_mnist_dataset('test', path)
    return X, y, X_test, y_test



## Training

10 epochs, batch size 128, Adam with learning rate decay, L2 regularization
(5e-4) and dropout (0.1) on both hidden layers.

In [9]:
# ══════════════════════════════════════════════════════════════════════
# MAIN — build, train, evaluate
# ══════════════════════════════════════════════════════════════════════

FASHION_MNIST_CLASSES = {
    0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'
}


download_fashion_mnist()
X, y, X_test, y_test = create_data_mnist('fashion_mnist_images')


keys = np.array(range(X.shape[0]))
np.random.shuffle(keys)
X = X[keys]
y = y[keys]


X = (X.reshape(X.shape[0], -1).astype(np.float32) - 127.5) / 127.5
X_test = (X_test.reshape(X_test.shape[0], -1).astype(np.float32) - 127.5) / 127.5


model = Model()

model.add(Layer_Dense(X.shape[1], 128,
                       weight_regularizer_l2=5e-4,
                       bias_regularizer_l2=5e-4))
model.add(Activation_ReLU())
model.add(Layer_Dropout(0.1))

model.add(Layer_Dense(128, 64,
                       weight_regularizer_l2=5e-4,
                       bias_regularizer_l2=5e-4))
model.add(Activation_ReLU())
model.add(Layer_Dropout(0.1))

model.add(Layer_Dense(64, 10))
model.add(Activation_Softmax())

model.set(
    loss=Loss_CategoricalCrossentropy(),
    optimizer=Optimizer_Adam(learning_rate=0.001, decay=5e-5),
    accuracy=Accuracy_Categorical()
)

model.finalize()

model.train(X, y,
             validation_data=(X_test, y_test),
             epochs=10,
             batch_size=128,
             print_every=200)


predictions = model.predict(X_test, batch_size=128)
predicted_classes = np.argmax(predictions, axis=1)

n_classes = 10
confusion = np.zeros((n_classes, n_classes), dtype=int)
for true_label, pred_label in zip(y_test, predicted_classes):
    confusion[true_label, pred_label] += 1

print('\nConfusion Matrix (rows = true label, cols = predicted label)')
print('     ' + ' '.join(f'{i:5d}' for i in range(n_classes)))
for i in range(n_classes):
    print(f'{i:3d}: ' + ' '.join(f'{confusion[i, j]:5d}' for j in range(n_classes)))

print('\nClass labels:')
for k, v in FASHION_MNIST_CLASSES.items():
    print(f'  {k}: {v}')


print('\nPer-class accuracy:')
for i in range(n_classes):
    class_acc = confusion[i, i] / confusion[i].sum()
    print(f'  {FASHION_MNIST_CLASSES[i]:15s}: {class_acc:.3f}')

Unzipping images...
Data ready.
epoch: 1
  step: 0, acc: 0.109, loss: 3.795 (data_loss: 3.253, reg_loss: 0.542), lr: 0.001
  step: 200, acc: 0.773, loss: 1.012 (data_loss: 0.553, reg_loss: 0.458), lr: 0.0009900990099009901
  step: 400, acc: 0.836, loss: 0.813 (data_loss: 0.427, reg_loss: 0.387), lr: 0.000980392156862745
  step: 468, acc: 0.812, loss: 0.933 (data_loss: 0.567, reg_loss: 0.365), lr: 0.0009771350400625367
  training,   acc: 0.779, loss: 0.977 (data_loss: 0.612, reg_loss: 0.365), lr: 0.0009771350400625367
  validation, acc: 0.838, loss: 0.440
epoch: 2
  step: 0, acc: 0.852, loss: 0.781 (data_loss: 0.416, reg_loss: 0.365), lr: 0.0009770873027505008
  step: 200, acc: 0.828, loss: 0.753 (data_loss: 0.441, reg_loss: 0.313), lr: 0.0009676326866321544
  step: 400, acc: 0.883, loss: 0.648 (data_loss: 0.377, reg_loss: 0.271), lr: 0.0009583592888974076
  step: 468, acc: 0.844, loss: 0.803 (data_loss: 0.544, reg_loss: 0.259), lr: 0.0009552466924583273
  training,   acc: 0.845, loss: 

## Results Analysis

**Overall:** 87.9% validation accuracy after 10 epochs — close to the
training accuracy (88.6%), indicating the model is not overfitting
significantly (L2 + dropout are doing their job).

**Per-class breakdown — the confusion matrix tells the real story:**

The model excels at visually distinct categories:
- Trouser: 96.6%
- Bag: 96.0%
- Ankle boot: 95.6%
- Sneaker: 94.7%

It struggles most with **Shirt (65.1%)** — by far the worst class. The
confusion matrix shows Shirt is most often misclassified as:
- T-shirt/top (131 times)
- Coat (80 times)
- Pullover (102 times)

This makes intuitive sense: Shirt, T-shirt/top, Pullover, and Coat are all
upper-body garments with overlapping silhouettes — a flattened 784-pixel
vector loses the spatial detail (collar shape, sleeve length, button
placement) that would distinguish them.

**Why this happens:** A dense network treats each of the 784 input pixels
as an independent feature with no notion of spatial relationships — pixel
(5, 10) and pixel (5, 11) are just two unrelated numbers to the first
layer. Convolutional layers solve exactly this problem by preserving
spatial structure and learning local patterns (edges, shapes) that
generalize across positions.

**Next step:** Implement `Layer_Conv2D` and `Layer_MaxPool2D` from scratch
to address this — convolution backward pass is itself a convolution with
flipped kernels, and pooling backprop routes gradients only to the
max-activated positions.